# Discord Persona Fine-Tuning: Mistral NeMo 12B

This notebook trains a QLoRA adapter on top of Mistral NeMo 12B to clone a specific Discord user's conversational style. It is heavily optimized to run on a free Google Colab T4 GPU (15GB VRAM).

### Key Optimizations Enabled:
* **4-bit Quantization (NF4):** Shrinks the 12B model down to ~6.5GB.
* **Gradient Checkpointing:** Trades computation time for massive VRAM savings.
* **Paged 8-bit Optimizer:** Pages optimizer states to system RAM to prevent OOM spikes.
* **Sequence Length Capping:** Limits context to 2048 tokens to keep attention matrices small.
* **AES-256 Encryption Support:** Protects your dataset and resulting model weights.

---

## Technical Decision Log & Optimization Notes

### 1. Hardware Constraint: T4 GPU and $BFloat16$
* **Context**: The Google Colab T4 GPU is based on the Turing architecture, which does not natively support the $BFloat16$ (BF16) datatype. 
* **Problem**: Mistral NeMo 12B defaults its internal configuration to $bfloat16$. Even with `fp16=True` enabled, base layers like word embeddings and the `lm_head` often remain in BF16. When the `GradScaler` encounters these gradients during the backward pass, it throws a `NotImplementedError`.
* **Decision**: 
    * Set `torch_dtype=torch.float16` during model loading and explicitly override `model.config.torch_dtype`.
    * **Scorched-Earth Sanitization**: Added a mandatory loop before training that scans every parameter. It forces all `requires_grad=True` parameters (LoRA adapters) to $float32$ and downcasts any lingering frozen $bfloat16$ base layers to $float16$ to ensure complete T4 compatibility.

### 2. Tokenizer Stability and Regex Conflicts
* **Context**: Mistral NeMo models have a known bug in their "fast" (Rust-based) tokenizer regex pattern.
* **Problem**: Passing `fix_mistral_regex=True` while using `use_fast=False` causes a `TypeError` because the "slow" tokenizer does not accept that keyword argument.
* **Decision**: 
    * **Strict Slow Tokenizer**: Set `use_fast=False`. The Python-based tokenizer is natively immune to the Rust regex bug, and removing the patch flag avoids library-level initialization conflicts.

### 3. Disk Persistence vs. VRAM Volatility
* **Context**: Loading and unzipping a 12B model takes ~20 minutes. Colab VRAM frequently fragments or deadlocks after a `NotImplementedError` or `OutOfMemoryError`, requiring a session restart.
* **Problem**: A standard session restart clears the VRAM but keeps the local `/content` disk intact. Original scripts ignored this, wasting 20 minutes re-extracting files that were already there.
* **Decision**: 
    * **State-Aware Extraction**: Updated the Model Retrieval cell to check for the existence of the `/content/model_local` directory. If populated, it bypasses all download/copy/extraction logic, reducing the wait time after a restart from 20 minutes to roughly 2 minutes.

### 4. VRAM Management and OOM Prevention
* **Context**: 12B parameter models are at the absolute limit of 15GB VRAM.
* **Decision**: 
    * **4-bit Quantization (NF4)**: Essential to fit the base weights (~6.5GB).
    * **Gradient Checkpointing**: Enabled to trade compute for memory by not storing all intermediate activations.
    * **Paged 8-bit Optimizer**: Offloads optimizer states to system RAM to prevent OOM spikes during weight updates.

In [ ]:
# ==========================================
# USER CONFIGURATION PARAMETERS
# ==========================================

# 1. Dataset Source Configuration
# If your zip is already in your Google Drive, provide the path here (fastest/safest method)
LOCAL_DRIVE_ZIP_PATH = '' # e.g., '/content/drive/MyDrive/dataset.zip'

# Otherwise, set the shared Google Drive link for your dataset zip here.
# The zip MUST contain the file structure: processed/dataset.jsonl
GDRIVE_SHARED_LINK = 'DRIVE_LINK_HERE'

# 1.5 AES-256 Encryption / Decryption Keys
# If your dataset is an AES-256 encrypted zip, provide the password here.
DECRYPTION_KEY = ""
# The output model will be zipped and encrypted with this key.
# If blank, it defaults to DECRYPTION_KEY. If both are blank, it saves unencrypted.
ENCRYPTION_KEY = ""

# 1.6 Hugging Face Authentication
# Paste your Hugging Face access token here to download the model
HF_TOKEN = ""

# 1.7 Model Source Configuration
# If your model is already in your Google Drive, provide the path here (folder or .zip)
LOCAL_DRIVE_MODEL_PATH = '' # e.g., '/content/drive/MyDrive/soulclone/Mistral-Nemo-Instruct-2407.zip'

# Otherwise, set the shared Google Drive link for your model zip here.
GDRIVE_MODEL_LINK = 'DRIVE_LINK_HERE'

# 2. Seed Control
USER_SEED = 42

# 3. Training Hyperparameters (T4 Optimized)
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1           # Keep at 1 to prevent OOM
GRAD_ACCUM_STEPS = 4     # Simulates a batch size of 4
EPOCHS = 1               # 1 to 3 is usually enough for style cloning
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32

In [ ]:
import os
import datetime
import time
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Setup Timestamped Directory
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_output_dir = f"/content/drive/MyDrive/soulclone/{timestamp}"
os.makedirs(run_output_dir, exist_ok=True)

# Global Logging Wrapper
master_log = ""
def log_print(*args, **kwargs):
    global master_log
    output = " ".join(map(str, args))
    print(output, **kwargs)
    master_log += output + "\n"

# Timing Wrapper
section_times = {}
global_start_time = time.time()

def start_timer(section_name):
    section_times[section_name] = {'start_time': time.time()}
    start_str = datetime.datetime.fromtimestamp(section_times[section_name]['start_time']).strftime('%H:%M:%S')
    log_print(f"\n[INFO] Started '{section_name}' at {start_str}")

def stop_timer(section_name):
    if section_name in section_times and 'start_time' in section_times[section_name]:
        end_t = time.time()
        section_times[section_name]['end_time'] = end_t
        duration = end_t - section_times[section_name]['start_time']
        section_times[section_name]['duration'] = duration
        end_str = datetime.datetime.fromtimestamp(end_t).strftime('%H:%M:%S')
        log_print(f"[INFO] Finished '{section_name}' at {end_str} (Took {duration:.2f}s)")

current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
log_print("\n" + "="*50)
log_print(f"RUN STARTED: {current_time}")
log_print(f"CONFIGURATION (LOCAL PATH): {LOCAL_DRIVE_ZIP_PATH}")
log_print(f"CONFIGURATION (DRIVE LINK): {GDRIVE_SHARED_LINK}")
log_print(f"AES ENCRYPTION ENABLED: {'Yes' if (ENCRYPTION_KEY or DECRYPTION_KEY) else 'No'}")
log_print(f"SEED: {USER_SEED}")
log_print(f"OUTPUT DIR: {run_output_dir}")
log_print("="*50 + "\n")

In [ ]:
start_timer("1. Install Dependencies")
!pip install -q transformers datasets peft trl bitsandbytes gdown accelerate pyzipper
stop_timer("1. Install Dependencies")

In [ ]:
start_timer("2. Data Download & Extraction")
import gdown
import pyzipper
import shutil

local_zip = '/content/dataset.zip'
local_extract_dir = '/content/local_data'

def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

# Fetch dataset file
if LOCAL_DRIVE_ZIP_PATH and os.path.exists(LOCAL_DRIVE_ZIP_PATH):
    log_print(f"Copying dataset locally from: {LOCAL_DRIVE_ZIP_PATH}")
    shutil.copy(LOCAL_DRIVE_ZIP_PATH, local_zip)
elif GDRIVE_SHARED_LINK != 'DRIVE_LINK_HERE':
    log_print("Downloading zip from Google Drive via gdown...")
    file_id = get_gdrive_id(GDRIVE_SHARED_LINK)
    download_url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(download_url, local_zip, quiet=False)
else:
    log_print("WARNING: No valid dataset source provided.")

# Extract dataset file
if os.path.exists(local_zip):
    os.makedirs(local_extract_dir, exist_ok=True)
    log_print("Extracting dataset...")
    try:
        if DECRYPTION_KEY:
            log_print("Using provided DECRYPTION_KEY for AES-256 decryption...")
            with pyzipper.AESZipFile(local_zip, 'r', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zip_ref:
                zip_ref.pwd = DECRYPTION_KEY.encode('utf-8')
                zip_ref.extractall(local_extract_dir)
        else:
            with pyzipper.AESZipFile(local_zip, 'r') as zip_ref:
                zip_ref.extractall(local_extract_dir)
    except Exception as e:
        log_print(f"Extraction failed (Check if key is missing or incorrect): {e}")

dataset_path = os.path.join(local_extract_dir, 'processed', 'dataset.jsonl')
if os.path.exists(dataset_path):
    log_print(f"Successfully located dataset at: {dataset_path}")
else:
    log_print(f"ERROR: Could not find dataset at {dataset_path}. Check your zip structure.")

stop_timer("2. Data Download & Extraction")

In [ ]:
start_timer("3. Model Retrieval & Extraction")
import os
import shutil
import zipfile
import gdown

local_model_zip = '/content/model_archive.zip'
local_model_extract_dir = '/content/model_local'

# Default fallback
model_id = "mistralai/Mistral-Nemo-Instruct-2407"

def get_gdrive_id(url):
    if '/d/' in url:
        return url.split('/d/')[1].split('/')[0]
    elif 'id=' in url:
        return url.split('id=')[1].split('&')[0]
    return url

# CRITICAL FIX: Check if model is already extracted to skip redundant work
if os.path.exists(local_model_extract_dir) and os.path.isdir(local_model_extract_dir) and len(os.listdir(local_model_extract_dir)) > 0:
    log_print(f"Model already extracted locally at: {local_model_extract_dir}. Skipping download and extraction.")
    model_id = local_model_extract_dir
else:
    if LOCAL_DRIVE_MODEL_PATH and os.path.exists(LOCAL_DRIVE_MODEL_PATH):
        if os.path.isdir(LOCAL_DRIVE_MODEL_PATH):
            log_print(f"Using unzipped local model directory: {LOCAL_DRIVE_MODEL_PATH}")
            model_id = LOCAL_DRIVE_MODEL_PATH
        elif LOCAL_DRIVE_MODEL_PATH.endswith('.zip'):
            log_print(f"Copying model zip locally from: {LOCAL_DRIVE_MODEL_PATH}")
            shutil.copy(LOCAL_DRIVE_MODEL_PATH, local_model_zip)
    elif GDRIVE_MODEL_LINK != 'DRIVE_LINK_HERE':
        log_print("Downloading model zip from Google Drive via gdown...")
        file_id = get_gdrive_id(GDRIVE_MODEL_LINK)
        download_url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(download_url, local_model_zip, quiet=False)
    else:
        log_print("No local model path or Drive link provided. Falling back to Hugging Face Hub.")

    # Extract if a zip was fetched
    if os.path.exists(local_model_zip) and model_id != LOCAL_DRIVE_MODEL_PATH:
        os.makedirs(local_model_extract_dir, exist_ok=True)
        log_print("Extracting model zip (this may take a few minutes)...")
        try:
            with zipfile.ZipFile(local_model_zip, 'r') as zip_ref:
                zip_ref.extractall(local_model_extract_dir)
            log_print("Model extraction successful.")
            model_id = local_model_extract_dir
            # Clean up zip to save space
            os.remove(local_model_zip)
        except Exception as e:
            log_print(f"Model zip extraction failed: {e}")
            log_print("Falling back to Hugging Face Hub.")
            model_id = "mistralai/Mistral-Nemo-Instruct-2407"

log_print(f"Final Model ID / Path resolved to: {model_id}")
stop_timer("3. Model Retrieval & Extraction")

In [ ]:
start_timer("4. Load Model and Tokenizer")
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

log_print("Executing aggressive memory clearing...")
# Delete all possible references from previous training runs
for var_name in ['model', 'tokenizer', 'trainer', 'training_args', 'peft_config']:
    if var_name in globals():
        log_print(f"Deleting {var_name} from globals...")
        del globals()[var_name]

# Force garbage collection
gc.collect()

# Clear CUDA cache and IPC
if torch.cuda.is_available():
    log_print("Emptying CUDA cache...")
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

if model_id == "mistralai/Mistral-Nemo-Instruct-2407":
    if HF_TOKEN:
        log_print("Authenticating with Hugging Face via config token...")
        login(HF_TOKEN)
    else:
        log_print("Warning: No HF_TOKEN provided in config. Download may hang.")
else:
    log_print("Loading model locally. Skipping Hugging Face authentication.")

log_print(f"Configuring 4 bit quantization for {model_id}...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

log_print("Loading Tokenizer...")
# Safely bypass the fast tokenizer regex bug by using the slow tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

log_print("Loading Model (this will take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

model.config.torch_dtype = torch.float16
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
stop_timer("4. Load Model and Tokenizer")

In [ ]:
start_timer("5. Prepare Dataset")
from datasets import load_dataset

log_print("Loading JSONL dataset into HuggingFace format...")
raw_dataset = load_dataset('json', data_files=dataset_path, split='train')

def apply_chat_template(examples):
    texts = []
    for msgs in examples['messages']:
        sys_msg = None
        conv = []
        for m in msgs:
            if m['role'] == 'system':
                sys_msg = m
            else:
                conv.append(m)

        # Merge consecutive messages from the same role to prevent alternating template errors
        merged_conv = []
        for m in conv:
            if not merged_conv:
                merged_conv.append(m.copy())
            elif merged_conv[-1]['role'] == m['role']:
                merged_conv[-1]['content'] += "\n" + m['content']
            else:
                merged_conv.append(m.copy())

        # Ensure the conversation starts with a 'user' message after the system prompt
        if merged_conv and merged_conv[0]['role'] == 'assistant':
            merged_conv.pop(0)

        final_msgs = ([sys_msg] if sys_msg else []) + merged_conv

        # Automatically maps your system/user/assistant tags to NeMo's native format
        formatted_text = tokenizer.apply_chat_template(final_msgs, tokenize=False, add_generation_prompt=False)
        texts.append(formatted_text)
    return {'text': texts}

log_print("Applying Mistral NeMo chat templates to data...")
# Added remove_columns to prevent SFTTrainer from processing the raw messages column
train_dataset = raw_dataset.map(apply_chat_template, batched=True, remove_columns=raw_dataset.column_names)
log_print(f"Prepared {len(train_dataset)} conversational examples.")
stop_timer("5. Prepare Dataset")

In [ ]:
start_timer("6. LoRA Configuration & Training")
import os
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

if not hasattr(model, "peft_config"):
    model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    model = get_peft_model(model, peft_config)

# CRITICAL FIX: The BFloat16 Exterminator (Backward Hooks)
log_print("Applying strict float32 backward hooks to guarantee T4 compatibility...")

# 1. Cast specific sensitive base layers to float32 to prevent numeric overflow
for name, module in model.named_modules():
    if any(keyword in name.lower() for keyword in ["norm", "lm_head", "embed_tokens"]):
        module.to(torch.float32)

# 2. Force LoRA adapters to float32 and intercept their gradients
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        # This hook intercepts the gradient the exact moment it is calculated during the backward pass
        # and forces it to float32, completely blinding the GradScaler from any BFloat16 type promotion.
        param.register_hook(lambda grad: grad.to(torch.float32) if grad is not None else None)

class LoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log_print(f"Step {state.global_step}: {logs}")

training_args = SFTConfig(
    output_dir=os.path.join(run_output_dir, "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    bf16=False, 
    max_grad_norm=0.3,
    warmup_steps=10,
    lr_scheduler_type="constant",
    seed=USER_SEED,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=training_args,
    callbacks=[LoggingCallback()]
)

log_print("Starting Training...")
trainer.train()
stop_timer("6. LoRA Configuration & Training")

In [ ]:
start_timer("7. Saving & Exporting")
import matplotlib.pyplot as plt
import pyzipper

# 1. Save Final Model Weights
final_model_path = os.path.join(run_output_dir, "final_adapter")
trainer.save_model(final_model_path)
log_print(f"Saved final LoRA adapter to: {final_model_path}")

# 1.5 Encrypt Output Model (if requested)
target_enc_key = ENCRYPTION_KEY if ENCRYPTION_KEY else DECRYPTION_KEY
if target_enc_key:
    log_print("Zipping and AES-256 encrypting the final model...")
    encrypted_zip_path = os.path.join(run_output_dir, "final_adapter_encrypted.zip")
    with pyzipper.AESZipFile(encrypted_zip_path, 'w', compression=pyzipper.ZIP_DEFLATED, encryption=pyzipper.WZ_AES) as zf:
        zf.setpassword(target_enc_key.encode('utf-8'))
        for root, _, files in os.walk(final_model_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=os.path.dirname(final_model_path))
                zf.write(file_path, arcname)
    log_print(f"Encrypted model saved to: {encrypted_zip_path}")
else:
    log_print("No encryption key provided. Skipping output encryption.")

# 2. Graph Training Loss
history = trainer.state.log_history
loss_steps = [x['step'] for x in history if 'loss' in x]
loss_values = [x['loss'] for x in history if 'loss' in x]

if loss_values:
    plt.figure(figsize=(10, 5))
    plt.plot(loss_steps, loss_values, label='Training Loss', color='blue')
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training Loss over Time')
    plt.legend()
    plt.grid(True)
    loss_graph_path = os.path.join(run_output_dir, f"training_loss_{timestamp}.png")
    plt.savefig(loss_graph_path, bbox_inches='tight')
    plt.show()
    log_print(f"Saved loss graph to: {loss_graph_path}")

# 3. Generate Summary Image
total_duration = time.time() - global_start_time
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')

y_pos = 0.95
ax.text(0.05, y_pos, "Mistral NeMo 12B - Fine-Tuning Summary", fontsize=18, fontweight='bold')
y_pos -= 0.1
ax.text(0.05, y_pos, "Model Configuration:", fontsize=14, fontweight='bold')
y_pos -= 0.05
ax.text(0.1, y_pos, f"Base Model: {model_id}", fontsize=12)
y_pos -= 0.05
ax.text(0.1, y_pos, f"LoRA Rank/Alpha: {LORA_R} / {LORA_ALPHA}", fontsize=12)
y_pos -= 0.05
ax.text(0.1, y_pos, f"Seed: {USER_SEED}", fontsize=12)
y_pos -= 0.1

ax.text(0.05, y_pos, "Execution Times:", fontsize=14, fontweight='bold')
y_pos -= 0.05
for section, times in section_times.items():
    if 'duration' in times:
        ax.text(0.1, y_pos, f"{section}: {times['duration']:.2f}s", fontsize=10, fontfamily='monospace')
        y_pos -= 0.04

y_pos -= 0.05
ax.text(0.05, y_pos, f"Total Execution Time: {total_duration:.2f}s", fontsize=14, fontweight='bold', color='darkgreen')

summary_img_path = os.path.join(run_output_dir, f"summary_stats_{timestamp}.png")
plt.savefig(summary_img_path, bbox_inches='tight', facecolor='white')
plt.show()
log_print(f"Saved summary image to {summary_img_path}")

# 4. Save Final Master Log
log_file_path = os.path.join(run_output_dir, f'compiled_training_log_{timestamp}.txt')
with open(log_file_path, 'w', encoding='utf-8') as f:
    f.write(master_log)
print(f"\nCompiled text log successfully saved to {log_file_path}")
stop_timer("7. Saving & Exporting")